In [10]:
# 下载文件并命名为 Jamming_Classifier.zip
# !wget -O Jamming_Classifier.zip "https://zenodo.org/records/3783969/files/Jamming_Classifier.zip?download=1"

# 解压下载的 ZIP 文件
# !unzip Jamming_Classifier.zip

In [ ]:
import os
import shutil
import random

# 類別列表
classes = ['DME', 'NB', 'NoJam', 'SingleAM', 'SingleChirp', 'SingleFM']

# 資料夾路徑
train_dir = 'Image_training_database'
test_dir = 'Image_testing_database'

# # 設定每個類別要挪多少張圖片過去
# num_to_move = 0  # 每類挪100張，可以自己改

# for label in classes:
#     train_label_dir = os.path.join(train_dir, label)
#     test_label_dir = os.path.join(test_dir, label)

#     # 如果測試集對應類別資料夾不存在，先創建
#     os.makedirs(test_label_dir, exist_ok=True)

#     # 列出所有訓練集下該類別的 bmp 文件
#     bmp_files = [f for f in os.listdir(train_label_dir) if f.endswith('.bmp')]

#     # 如果該類文件少於要挪的數量，取全部
#     move_files = random.sample(bmp_files, min(num_to_move, len(bmp_files)))

#     # 開始移動
#     for file in move_files:
#         src_path = os.path.join(train_label_dir, file)
#         dst_path = os.path.join(test_label_dir, file)
#         shutil.move(src_path, dst_path)

#     print(f"{label}：從訓練集移動了 {len(move_files)} 張圖片到測試集。")

DME：從訓練集移動了 100 張圖片到測試集。
NB：從訓練集移動了 100 張圖片到測試集。
NoJam：從訓練集移動了 100 張圖片到測試集。
SingleAM：從訓練集移動了 100 張圖片到測試集。
SingleChirp：從訓練集移動了 100 張圖片到測試集。
SingleFM：從訓練集移動了 100 張圖片到測試集。


In [12]:
import cv2
import numpy as np

img_size = 256  # 目标图像大小

def load_dataset_cv2(folder_path,X_num, y_num):
    X = []
    y = []
    error_files = []
    for idx, label in enumerate(classes):
        label_dir = os.path.join(folder_path, label)
        if not os.path.exists(label_dir):
            print(f"警告：{label_dir} 文件夹不存在，跳过")
            continue
        for file in os.listdir(label_dir):
            if file.endswith('.bmp'):
                img_path = os.path.join(label_dir, file)
                # 以灰度模式读取图像
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"跳过无法读取的文件：{img_path}")
                    error_files.append(img_path)
                    continue
                # 缩放图像到 256x256
                img = cv2.resize(img, (img_size, img_size))
                # 像素值归一化到 [0,1]
                img = img.astype(np.float32) / 255.0
                X.append(img)
                y.append(idx)
                if len(X) >= X_num and len(y) >= y_num:
                    break
    X = np.array(X)
    y = np.array(y)
    print(f"成功读入 {len(X)} 张图像，跳过 {len(error_files)} 个无法读取的文件")
    return X, y, error_files

In [13]:
# 分别加载训练集和测试集数据
X_train, y_train, train_errors = load_dataset_cv2(train_dir, 1000, 1000)
X_test, y_test, test_errors = load_dataset_cv2(test_dir, 1000, 1000)

成功读入 1005 张图像，跳过 0 个无法读取的文件
成功读入 1005 张图像，跳过 0 个无法读取的文件


In [14]:
# 统计每个数据集每类图像的数量，确保数据分布符合预期
def count_images(folder_path, dataset_name):
    print(f"\n{dataset_name} 圖片數量統計：")
    for label in classes:
        label_dir = os.path.join(folder_path, label)
        if not os.path.exists(label_dir):
            print(f"類別 {label} 不存在，跳過")
            continue
        count = len([file for file in os.listdir(label_dir) if file.endswith('.bmp')])
        print(f"{label}: {count} 張")

# 分別統計訓練集與測試集
count_images('Image_training_database', '訓練集')
count_images('Image_testing_database', '測試集')


訓練集 圖片數量統計：
DME: 9600 張
NB: 9600 張
NoJam: 9600 張
SingleAM: 9600 張
SingleChirp: 9600 張
SingleFM: 9600 張

測試集 圖片數量統計：
DME: 10000 張
NB: 10000 張
NoJam: 10000 張
SingleAM: 10000 張
SingleChirp: 10000 張
SingleFM: 10000 張


In [15]:
# 将灰度图像增加一个通道维度 (变为 HxW×1)
X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)
print("扩展通道维度后：")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

# 将单通道灰度复制成3通道 (RGB)
X_train_rgb = np.repeat(X_train, repeats=3, axis=-1)
X_test_rgb = np.repeat(X_test, repeats=3, axis=-1)
print("\n转为RGB 3通道后：")
print("X_train_rgb shape:", X_train_rgb.shape)
print("X_test_rgb shape:", X_test_rgb.shape)

扩展通道维度后：
X_train shape: (1005, 256, 256, 1)
X_test shape: (1005, 256, 256, 1)

转为RGB 3通道后：
X_train_rgb shape: (1005, 256, 256, 3)
X_test_rgb shape: (1005, 256, 256, 3)


In [16]:
import torch
import torch.nn as nn
from torchvision import models

# 使用预训练的 MobileNetV2 模型
base_model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
# 替换最后的分类器，使输出类别数等于len(classes)
# MobileNetV2的特征层输出维度为1280，我们去除原有的线性层，新增一个线性层输出6类
base_model.classifier = nn.Linear(base_model.classifier[1].in_features, len(classes))

# 冻结特征提取层的参数（卷积和BN层）
for param in base_model.features.parameters():
    param.requires_grad = False
# 同时，将BN层设置为评估模式，以避免在训练过程中更新其统计量
for m in base_model.features.modules():
    if isinstance(m, nn.BatchNorm2d):
        m.eval()

# 查看模型结构
print(base_model)

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [17]:
# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()  # 等价于 sparse_categorical_crossentropy
optimizer = torch.optim.Adam(base_model.parameters(), lr=0.001)

# 将模型移动到 GPU（如果有可用的CUDA设备）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model.to(device)

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [18]:
from sklearn.model_selection import train_test_split

# 将数据转换为numpy数组（此处实际上已经是numpy数组，无需特殊转换）
X_np = X_train_rgb  
y_np = y_train

# 按20%划分验证集
X_train_new, X_val, y_train_new, y_val = train_test_split(
    X_np, y_np,
    test_size=0.2,
    random_state=42,
    stratify=y_np
)
print("训练集划分后：")
print("X_train_new shape:", X_train_new.shape)
print("X_val shape:", X_val.shape)

# 转换分割后的数据为 PyTorch 张量，并调整维度顺序 (N,H,W,C)->(N,C,H,W)
X_train_tensor = torch.from_numpy(X_train_new).permute(0, 3, 1, 2).float()
X_val_tensor = torch.from_numpy(X_val).permute(0, 3, 1, 2).float()
X_test_tensor = torch.from_numpy(X_test_rgb).permute(0, 3, 1, 2).float()
y_train_tensor = torch.from_numpy(y_train_new).long()
y_val_tensor = torch.from_numpy(y_val).long()
y_test_tensor = torch.from_numpy(y_test).long()

# 使用DataLoader封装数据集
from torch.utils.data import TensorDataset, DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [ ]:
epochs = 30
patience = 5
best_val_loss = float('inf')
best_state_dict = None
epochs_no_improve = 0

for epoch in range(1, epochs+1):
    base_model.train()  # 切换模型到训练模式
    train_loss_sum = 0.0
    train_correct = 0
    total_train = 0

    # 训练批次循环
    for X_batch, y_batch in train_loader:
        # 将数据加载到计算设备
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        # 前向传播
        outputs = base_model(X_batch)
        loss = criterion(outputs, y_batch)
        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # 统计训练损失和准确度
        train_loss_sum += loss.item() * y_batch.size(0)
        _, pred_labels = torch.max(outputs, 1)
        train_correct += (pred_labels == y_batch).sum().item()
        total_train += y_batch.size(0)
    # 计算平均训练损失和准确率
    avg_train_loss = train_loss_sum / total_train
    train_accuracy = train_correct / total_train

    # 验证阶段
    base_model.eval()  # 切换模型到评估模式
    val_loss_sum = 0.0
    val_correct = 0
    total_val = 0
    # 在验证集上不需要计算梯度
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = base_model(X_batch)
            loss = criterion(outputs, y_batch)
            # 累计验证损失和准确度
            val_loss_sum += loss.item() * y_batch.size(0)
            _, pred_labels = torch.max(outputs, 1)
            val_correct += (pred_labels == y_batch).sum().item()
            total_val += y_batch.size(0)
    avg_val_loss = val_loss_sum / total_val
    val_accuracy = val_correct / total_val

    # 输出本轮训练的结果
    print(f"Epoch {epoch}/{epochs} - "
          f"loss: {avg_train_loss:.4f} - accuracy: {train_accuracy:.4f} - "
          f"val_loss: {avg_val_loss:.4f} - val_accuracy: {val_accuracy:.4f}")

    # Early Stopping 检查：若验证损失改善，则保存最佳模型权重
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_state_dict = base_model.state_dict()  # 保存当前最佳状态
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("验证集 loss 多次没有改善，提前停止训练。")
            if best_state_dict is not None:
                base_model.load_state_dict(best_state_dict)  # 恢复最佳模型权重
            break

In [ ]:
base_model.eval()  # 模型设为评估模式
test_correct = 0
total_test = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        outputs = base_model(X_batch)
        _, pred_labels = torch.max(outputs, 1)
        test_correct += (pred_labels == y_batch).sum().item()
        total_test += y_batch.size(0)
test_accuracy = test_correct / total_test
print(f"测试集上的准确率: {test_accuracy * 100:.2f}%")